# Project 1: Data Cleaning & Preparation
**DecodeLabs Data Analytics Internship**

Goal: clean a raw e-commerce orders dataset  by handling missing values, duplicate identifiers, and inconsistent formatting.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'C:\Users\Admin\Downloads\Dataset for Data Analytics - Sheet1.csv')
df

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,1/4/2023,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,8/23/2024,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2/27/2024,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,10/15/2023,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,5/8/2025,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,6/20/2024,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,3/4/2024,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,NaN,Facebook,1325.06
1197,ORD201197,7/13/2023,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,8/22/2024,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


In [3]:
df.shape

(1200, 14)

In [4]:
df.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

In [5]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,1/4/2023,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,8/23/2024,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2/27/2024,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,10/15/2023,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,5/8/2025,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   object 
 1   Date             1200 non-null   object 
 2   CustomerID       1200 non-null   object 
 3   Product          1200 non-null   object 
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   object 
 7   PaymentMethod    1200 non-null   object 
 8   OrderStatus      1200 non-null   object 
 9   TrackingNumber   1200 non-null   object 
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       891 non-null    object 
 12  ReferralSource   1200 non-null   object 
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), object(10)
memory usage: 131.4+ KB


## Initial Inspection
Before making any changes, audit the raw data for missing values and duplicates.

In [7]:
df.isnull().sum() #Missing values per column


OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [5]:
print("Full duplicate rows:",df.duplicated().sum())

Full duplicate rows: 0


In [9]:
print("Duplicate OrderIDs:",df.duplicated(subset=['OrderID']).sum())

Duplicate OrderIDs: 0


## Phase 1: Strategic Imputation

`CouponCode` has missing values. Since it's a categorical (text) column, mean/median don't apply, mode is the only  option left.

**Decision:** the mode (`FREESHIP`) is deliberately *not* used to fill the gaps. Doing so would fabricate a discount the customer never received. The missing values almost certainly represent orders where **no coupon was applied** — a legitimate business state, not an error. An explicit `'No Coupon'` label is used instead of a statistical guess.

In [10]:
df['CouponCode']

0         SAVE10
1         SAVE10
2       FREESHIP
3         SAVE10
4         SAVE10
          ...   
1195    FREESHIP
1196         NaN
1197    FREESHIP
1198    WINTER15
1199      SAVE10
Name: CouponCode, Length: 1200, dtype: object

In [11]:
df['CouponCode'].mode()

0    FREESHIP
Name: CouponCode, dtype: object

In [12]:
df['CouponCode'].value_counts()

CouponCode
FREESHIP    313
WINTER15    292
SAVE10      286
Name: count, dtype: int64

In [14]:
# Fill missing coupon codes with an explicit label
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')

print(df['CouponCode'].isnull().sum())

0


In [15]:
df['CouponCode'].value_counts()

CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

In [16]:
Dup_orderid = df.duplicated(subset=['OrderID']).sum()
print(f"Duplicate Order IDs:",Dup_orderid)

Dup_TrackingNumber = df.duplicated(subset=['TrackingNumber']).sum()
print(f"Duplicate Tracking Numbers:",Dup_TrackingNumber)

Dup_fullrow = df.duplicated().sum()
print(f"Duplicate Full row:",Dup_fullrow)

Duplicate Order IDs: 0
Duplicate Tracking Numbers: 0
Duplicate Full row: 0


## Phase 3: Standardization 

Convert `Date` to a true datetime type (ISO 8601) and round currency columns (`UnitPrice`, `TotalPrice`) to 2 decimal places. `Quantity` and `ItemsInCart` are intentionally left as whole numbers,they represent counts, not currency.

In [17]:
#Check Current data type
print(df['Date'].dtype)

object


In [18]:
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
print(df['Date'].dtype)
print(df['Date'].head())

datetime64[ns]
0   2023-01-04
1   2024-08-23
2   2024-02-27
3   2023-10-15
4   2025-05-08
Name: Date, dtype: datetime64[ns]


In [19]:
print(df['UnitPrice'].dtype)
print(df['TotalPrice'].dtype)

float64
float64


In [20]:
df['UnitPrice'] = df['UnitPrice'].round(2)
df['UnitPrice']

0       570.62
1       151.35
2       550.68
3       273.19
4       626.01
         ...  
1195    107.04
1196    662.53
1197    436.84
1198    262.52
1199    560.58
Name: UnitPrice, Length: 1200, dtype: float64

In [21]:
df['TotalPrice'] = df['TotalPrice'].round(2)
df['TotalPrice']

0       2853.10
1        302.70
2       2753.40
3        273.19
4       2504.04
         ...   
1195     107.04
1196    1325.06
1197     873.68
1198    1050.08
1199    2242.32
Name: TotalPrice, Length: 1200, dtype: float64

In [22]:
df.head(20)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,No Coupon,Email,268.56
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


In [24]:
df.to_csv('Cleaned_dataset.csv',index=False)